## Exercise — NorthBank Portfolio KRI Calculator

You will implement two of NorthBank's five portfolio KRIs as production-grade Python functions, then render the 5-KRI dashboard.

**KRIs to implement (Part 4 of the exercise):**
1. `compute_subgroup_fnr_delta()` — KRI #1 (fraud model, region × income subgroups).
2. `compute_drift_score()` — KRI #2 (credit-decisioning model, max PSI across features).

The other three KRIs (anomalous query rate, vendor SLA breach rate, threat-model coverage) ship as *design specs only* in `kri_definitions.xlsx`. You don't implement them in this exercise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

DATA = Path("data")
np.random.seed(42)

## 1. KRI #1 — Subgroup FNR delta (Fraud model)

In [ ]:
def status_band(value, green, red):
    """Return 'green' / 'amber' / 'red' based on numeric thresholds.

    Args:
        value: the metric value
        green: upper bound of green band (inclusive)
        red:   upper bound of amber band (inclusive)
    """
    if value <= green: return "green"
    if value <= red:   return "amber"
    return "red"


def compute_subgroup_fnr_delta(df, label_col="label", pred_col="prediction",
                               group_cols=("region", "income_tier"),
                               green=0.05, red=0.10,
                               owner="Fraud Model Owner"):
    """Compute the worst-case subgroup FNR gap on the predictions DataFrame.

    Args:
        df:        DataFrame with label_col, pred_col, and the group_cols.
        label_col: name of the ground-truth column (1 = positive case).
        pred_col:  name of the model-prediction column (1 = predicted positive).
        group_cols: tuple of subgroup column names (e.g. ("region", "income_tier")).
        green / red: KRI threshold bounds.
        owner:     human owner string for the KRI return shape.

    Returns:
        {value, status, owner, timestamp}.
        - value  = max(FNR) - min(FNR) across all subgroups (rounded to 4 dp).
        - status = "green" / "amber" / "red" via status_band().
    """
    # TODO Step 1: group df by the group_cols.
    # TODO Step 2: for each subgroup, compute FNR = (label==1 & pred==0).sum() / max(1, (label==1).sum()).
    # TODO Step 3: value = float(fnr.max() - fnr.min()), rounded to 4 decimal places.
    # TODO Step 4: return the standard KRI dict shape (value / status / owner / timestamp).
    pass

### Test KRI #1 against the fraud_predictions.csv fixture

In [ ]:
fraud = pd.read_csv(DATA / "fraud_predictions.csv", parse_dates=["date"])
result_kri1 = compute_subgroup_fnr_delta(fraud)
print(result_kri1)

# Expected (deterministic with seed=42):
#   value  ≈ 0.1053
#   status = "red"
assert result_kri1 is not None and "value" in result_kri1, "compute_subgroup_fnr_delta returned None — implement the function"
assert abs(result_kri1["value"] - 0.1053) < 0.005, f"FNR delta mismatch: got {result_kri1['value']}"
assert result_kri1["status"] == "red", f"Status mismatch: got {result_kri1['status']}"
print("KRI #1 PASS")

## 2. KRI #2 — Feature drift score (Credit-decisioning model, max PSI)

In [ ]:
def population_stability_index(baseline, current, bins=10):
    """Compute the PSI (population stability index) between two 1-D distributions.

    Args:
        baseline: 1-D iterable (training-time or last-quarter values).
        current:  1-D iterable (current-period values).
        bins:     number of equal-width bins used for the histogram.

    Returns:
        Float PSI value. Higher = more drift. Rule of thumb: < 0.1 stable;
        0.1-0.25 some shift; > 0.25 significant.
    """
    baseline = np.asarray(baseline, dtype=float)
    current = np.asarray(current, dtype=float)
    edges = np.linspace(min(baseline.min(), current.min()),
                        max(baseline.max(), current.max()),
                        bins + 1)
    b_hist, _ = np.histogram(baseline, bins=edges)
    c_hist, _ = np.histogram(current, bins=edges)
    b_pct = (b_hist + 1e-6) / (b_hist.sum() + 1e-6 * bins)
    c_pct = (c_hist + 1e-6) / (c_hist.sum() + 1e-6 * bins)
    return float(np.sum((c_pct - b_pct) * np.log(c_pct / b_pct)))


def compute_drift_score(df, snapshot_col="snapshot",
                        feature_cols=("debt_to_income", "credit_age_years",
                                      "recent_inquiries", "utilization"),
                        green=0.10, red=0.25,
                        owner="Credit Model Owner"):
    """KRI = max PSI across the named feature columns between baseline + current snapshots.

    Args:
        df: DataFrame with snapshot_col labels in {"baseline", "current"} and the feature_cols.
        feature_cols: which columns to compute PSI for.
        green / red: KRI threshold bounds.
        owner: KRI owner string.

    Returns:
        {value, status, owner, timestamp} where value = max PSI across features.
    """
    # TODO Step 1: split df into baseline = df[df[snapshot_col]=="baseline"] and current = df[df[snapshot_col]=="current"].
    # TODO Step 2: for each col in feature_cols, compute population_stability_index(baseline[col], current[col]).
    # TODO Step 3: value = round(max(psi values), 4).
    # TODO Step 4: return the standard KRI dict shape.
    pass

### Test KRI #2 against the credit_drift_features.csv fixture

In [ ]:
drift = pd.read_csv(DATA / "credit_drift_features.csv")
result_kri2 = compute_drift_score(drift)
print(result_kri2)

# Expected (deterministic with seed=42):
#   value  > 0.25 (utilization + DTI features both drift heavily)
#   status = "red"
assert result_kri2 is not None and "value" in result_kri2, "compute_drift_score returned None — implement the function"
assert result_kri2["value"] >= 0.25, f"Drift score should be > 0.25 (red); got {result_kri2['value']}"
assert result_kri2["status"] == "red", f"Status mismatch: got {result_kri2['status']}"
print("KRI #2 PASS")

## 3. Render the 5-KRI dashboard

In [ ]:
# 5-KRI dashboard. KRIs 1 + 2 use computed values from your calculators above;
# KRIs 3 / 4 / 5 are design-spec placeholders (see kri_definitions.xlsx).

dashboard = [
    {"kri": "Subgroup FNR delta (Fraud)",          "value": result_kri1["value"], "status": result_kri1["status"], "amber": 0.05,  "red": 0.10},
    {"kri": "Feature drift PSI (Credit)",          "value": result_kri2["value"], "status": result_kri2["status"], "amber": 0.10,  "red": 0.25},
    {"kri": "Anomalous query rate (LLM)",          "value": None,                  "status": "design-spec",         "amber": 0.005, "red": 0.02},
    {"kri": "Vendor SLA breach rate (FM vendors)", "value": None,                  "status": "design-spec",         "amber": 0.001, "red": 0.005},
    {"kri": "% prod models w/ current threat model","value": None,                 "status": "design-spec",         "amber": 0.90,  "red": 0.70},
]

# TODO Step 5: render a horizontal bar chart with one bar per KRI.
#   - Use status to pick bar color (green / amber / red / gray for design-spec).
#   - For each KRI with a numeric value, draw vertical lines at amber + red thresholds.
#   - Title: "NorthBank — AI Security KRI Dashboard"
#   - Save the figure to "kri_dashboard.png" (250 dpi).

## Key takeaway

A board dashboard is a communication artifact, but a KRI is a Python function. The dashboard above ships with KRIs 1 + 2 live and KRIs 3 / 4 / 5 as placeholders — the AIRB sees the intended portfolio shape on day one, and the next engineering sprint wires the remaining three calculators against the same return shape.

If the function is right, the dashboard is trivial; if the function is wrong, no amount of styling fixes the metric.